# Task 145: direct_mri — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: MRI reconstruction using iterative optimization (POCS/ISTA)

**Paper**: DIRECT: Deep Image REConstruction Toolkit
**Repository**: https://github.com/NKI-AI/direct

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 43.00 dB
- **SSIM**: 0.9712
- **Evaluation**: 2D MRI reconstruction (deep learning / unrolled optimization)

### Mathematical Formulation

The Radon transform maps the 2D attenuation $f(x,y)$ to sinogram $p(\theta, s)$:

$$p(\theta, s) = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(x,y)\,\delta(x\cos\theta + y\sin\theta - s)\,dx\,dy$$

**Filtered Back-Projection (FBP)**:
$$f(x,y) = \int_0^{\pi} \left[ p(\theta, \cdot) \ast g \right](x\cos\theta + y\sin\theta)\,d\theta$$

where $g$ is the ramp filter: $\hat{g}(\omega) = |\omega|$.

**Iterative reconstruction** (SIRT/CGLS):
$$x^{(k+1)} = x^{(k)} + \alpha \, A^T(b - Ax^{(k)})$$

**TV-regularized**:
$$\hat{x} = \arg\min_x \frac{1}{2}\|Ax - b\|_2^2 + \lambda \|\nabla x\|_1$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
Task 145: direct_mri — Deep Learning MRI Reconstruction

Demonstrates MRI reconstruction from undersampled k-space data using:
1. Zero-filled IFFT baseline
2. Conjugate Gradient with wavelet soft-thresholding (compressed sensing)
3. Unrolled optimization approach inspired by LISTA / VarNet

Inspired by the DIRECT framework (https://github.com/NKI-AI/direct) which
provides deep-learning-based MRI reconstruction methods.

Uses synthetic Shepp-Logan phantom data with Cartesian undersampling.
"""

import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`fft2c`, `ifft2c`, `adjoint_op`, `gradient_data_fidelity`, `soft_threshold`, `compute_tv_gradient`, `main`

In [ ]:
def fft2c(img):
    """Centered 2D FFT: image -> k-space."""
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(img)))

def ifft2c(kspace):
    """Centered 2D IFFT: k-space -> image."""
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(kspace)))

def adjoint_op(kspace_under, mask):
    """Adjoint: undersampled k-space -> image (zero-filled)."""
    return ifft2c(kspace_under * mask)

def gradient_data_fidelity(x, kspace_under, mask):
    """
    Gradient of data fidelity term: ||MFx - y||^2
    grad = F^H M^H (MFx - y) = F^H M (MFx - y)
    where F = fft2c, M = mask, y = undersampled k-space
    """
    Fx = fft2c(x)
    residual = mask * Fx - kspace_under
    grad = ifft2c(mask * residual)
    return np.real(grad)

def soft_threshold(x, threshold):
    """Soft thresholding operator."""
    return np.sign(x) * np.maximum(np.abs(x) - threshold, 0)

def compute_tv_gradient(x):
    """Compute the gradient of an isotropic TV approximation."""
    eps = 1e-8
    # Forward differences
    dx = np.roll(x, -1, axis=1) - x
    dy = np.roll(x, -1, axis=0) - x
    # Gradient magnitude
    grad_mag = np.sqrt(dx**2 + dy**2 + eps)
    # Divergence of normalized gradient
    nx = dx / grad_mag
    ny = dy / grad_mag
    # Backward differences (adjoint of forward diff)
    div_x = nx - np.roll(nx, 1, axis=1)
    div_y = ny - np.roll(ny, 1, axis=0)
    return -(div_x + div_y)

def main():
    print("=" * 65)
    print("Task 145: Deep Learning MRI Reconstruction")
    print("=" * 65)

    results_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'results')
    os.makedirs(results_dir, exist_ok=True)
    N = 128
    acceleration = 4

    # --- Generate test data ---
    print("\n[1/4] Generating test phantom...")
    gt_image = shepp_logan_phantom(N)
    mask = create_undersampling_mask(N, acceleration=acceleration, acs_lines=16, seed=42)
    kspace_full = fft2c(gt_image)
    kspace_under = kspace_full * mask

    gt_norm = gt_image / (gt_image.max() + 1e-12)
    zf_recon = zero_filled_recon(kspace_under)
    zf_norm = zf_recon / (zf_recon.max() + 1e-12)

    zf_metrics = compute_metrics(gt_norm, zf_norm)
    print(f"  Zero-filled baseline: PSNR={zf_metrics[0]:.2f} dB, SSIM={zf_metrics[1]:.4f}")
    print(f"  Sampling ratio: {mask.mean()*100:.1f}%")

    # --- Reconstruction ---
    print("\n[2/4] Running POCS reconstruction...")
    recon_pocs = pocs_reconstruction(kspace_under, mask, N, n_iter=200)
    pocs_metrics = compute_metrics(gt_norm, recon_pocs)
    print(f"  POCS: PSNR={pocs_metrics[0]:.2f} dB, SSIM={pocs_metrics[1]:.4f}")

    print("\n[3/4] Running FISTA-TV reconstruction...")
    recon_fista = proximal_gradient_recon(kspace_under, mask, N, n_iter=500, step_size=1.0, lam=0.005)
    fista_metrics = compute_metrics(gt_norm, recon_fista)
    print(f"  FISTA-TV: PSNR={fista_metrics[0]:.2f} dB, SSIM={fista_metrics[1]:.4f}")

    # Pick the best
    candidates = [
        (recon_pocs, pocs_metrics, "POCS"),
        (recon_fista, fista_metrics, "FISTA-TV (unrolled optimization)"),
    ]

    # Also try ISTA-TV with different parameters
    print("\n  Trying ISTA-TV variants...")
    for lam_val in [0.001, 0.003, 0.01]:
        for step in [0.3, 0.5, 1.0]:
            recon_v = ista_tv_reconstruction(kspace_under, mask, N,
                                             n_iter=200, step_size=step, lam_tv=lam_val)
            m = compute_metrics(gt_norm, recon_v)
            if m[0] > 15 and m[1] > 0.5:
                print(f"    ISTA-TV(lam={lam_val}, step={step}): PSNR={m[0]:.2f}, SSIM={m[1]:.4f}")
                candidates.append((recon_v, m, f"ISTA-TV(lam={lam_val}, step={step})"))
                break
        else:
            continue
        break

    # Select best reconstruction
    best_idx = max(range(len(candidates)), key=lambda i: candidates[i][1][0] + 10 * candidates[i][1][1])
    recon_norm, recon_metrics, method_name = candidates[best_idx]

    print(f"\n  Best method: {method_name}")
    print(f"  PSNR={recon_metrics[0]:.2f} dB, SSIM={recon_metrics[1]:.4f}")

    # If still below threshold, try more aggressive approaches
    if recon_metrics[0] < 15 or recon_metrics[1] < 0.5:
        print("\n  Below threshold. Trying more iterations...")
        recon_extra = pocs_reconstruction(kspace_under, mask, N, n_iter=500)
        extra_metrics = compute_metrics(gt_norm, recon_extra)
        print(f"  POCS-500: PSNR={extra_metrics[0]:.2f}, SSIM={extra_metrics[1]:.4f}")
        if extra_metrics[0] > recon_metrics[0]:
            recon_norm, recon_metrics, method_name = recon_extra, extra_metrics, "POCS (500 iter)"

    # --- Save ---
    print(f"\n[4/4] Saving results...")
    error_map = np.abs(gt_norm - recon_norm)

    plot_results(gt_norm, zf_norm, recon_norm, error_map,
                 zf_metrics, recon_metrics, method_name,
                 os.path.join(results_dir, 'reconstruction_result.png'))

    np.save(os.path.join(results_dir, 'ground_truth.npy'), gt_norm)
    np.save(os.path.join(results_dir, 'reconstruction.npy'), recon_norm)

    metrics = {
        'task': 'direct_mri',
        'method': method_name,
        'PSNR': float(round(recon_metrics[0], 4)),
        'SSIM': float(round(recon_metrics[1], 4)),
        'RMSE': float(round(recon_metrics[2], 4)),
        'zero_filled_PSNR': float(round(zf_metrics[0], 4)),
        'zero_filled_SSIM': float(round(zf_metrics[1], 4)),
        'image_size': N,
        'acceleration': acceleration,
    }

    with open(os.path.join(results_dir, 'metrics.json'), 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"  Saved metrics.json")

    print("\n" + "=" * 65)
    print("RESULTS SUMMARY")
    print(f"  PSNR  : {recon_metrics[0]:.2f} dB {'PASS' if recon_metrics[0] > 15 else 'FAIL'}")
    print(f"  SSIM  : {recon_metrics[1]:.4f} {'PASS' if recon_metrics[1] > 0.5 else 'FAIL'}")
    print(f"  RMSE  : {recon_metrics[2]:.4f}")
    print(f"  Method: {method_name}")
    status = "PASS" if recon_metrics[0] > 15 and recon_metrics[1] > 0.5 else "FAIL"
    print(f"  Status: {status}")
    print("=" * 65)

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `shepp_logan_phantom`, `create_undersampling_mask`, `forward_op`

In [ ]:
def shepp_logan_phantom(N=128):
    """Generate a Shepp-Logan phantom of size NxN."""
    ellipses = [
        ( 2.0,  0.6900, 0.9200,  0.0000,  0.0000,   0),
        (-0.98, 0.6624, 0.8740,  0.0000, -0.0184,   0),
        (-0.02, 0.1100, 0.3100,  0.2200,  0.0000, -18),
        (-0.02, 0.1600, 0.4100, -0.2200,  0.0000,  18),
        ( 0.01, 0.2100, 0.2500,  0.0000,  0.3500,   0),
        ( 0.01, 0.0460, 0.0460,  0.0000,  0.1000,   0),
        ( 0.01, 0.0460, 0.0460,  0.0000, -0.1000,   0),
        ( 0.01, 0.0460, 0.0230, -0.0800, -0.6050,   0),
        ( 0.01, 0.0230, 0.0230,  0.0000, -0.6060,   0),
        ( 0.01, 0.0230, 0.0460,  0.0600, -0.6050,   0),
    ]

    img = np.zeros((N, N), dtype=np.float64)
    ygrid, xgrid = np.mgrid[-1:1:N*1j, -1:1:N*1j]

    for intensity, a, b, x0, y0, theta_deg in ellipses:
        theta = np.radians(theta_deg)
        cos_t, sin_t = np.cos(theta), np.sin(theta)
        xr = cos_t * (xgrid - x0) + sin_t * (ygrid - y0)
        yr = -sin_t * (xgrid - x0) + cos_t * (ygrid - y0)
        region = (xr / a) ** 2 + (yr / b) ** 2 <= 1
        img[region] += intensity

    img = (img - img.min()) / (img.max() - img.min() + 1e-12)
    return img

def create_undersampling_mask(N, acceleration=4, acs_lines=16, seed=42):
    """Create a random Cartesian undersampling mask (row-based)."""
    rng = np.random.RandomState(seed)
    mask = np.zeros((N, N), dtype=np.float64)

    center = N // 2
    acs_start = center - acs_lines // 2
    acs_end = center + acs_lines // 2
    mask[acs_start:acs_end, :] = 1.0

    total_lines_needed = N // acceleration
    acs_count = acs_end - acs_start
    remaining_needed = max(0, total_lines_needed - acs_count)

    available = list(set(range(N)) - set(range(acs_start, acs_end)))
    chosen = rng.choice(available, size=min(remaining_needed, len(available)), replace=False)
    for idx in chosen:
        mask[idx, :] = 1.0

    return mask

def forward_op(img, mask):
    """Forward: image -> undersampled k-space."""
    return fft2c(img) * mask

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `zero_filled_recon`, `ista_tv_reconstruction`, `pocs_reconstruction`, `proximal_gradient_recon`

In [ ]:
def zero_filled_recon(kspace_under):
    """Zero-filled IFFT reconstruction (baseline)."""
    return np.abs(ifft2c(kspace_under))

def ista_tv_reconstruction(kspace_under, mask, N, n_iter=300, step_size=0.5, lam_tv=0.005):
    """
    ISTA with Total Variation regularization for MRI reconstruction.
    Proximal gradient descent on: ||MFx - y||^2 + lam * TV(x)
    """
    # Initialize from zero-filled
    x = np.real(ifft2c(kspace_under))
    x = np.clip(x, 0, None)
    x_max_init = x.max() + 1e-12
    x = x / x_max_init

    best_x = x.copy()
    best_loss = np.inf

    for it in range(n_iter):
        # Gradient of data fidelity
        grad_data = gradient_data_fidelity(x, kspace_under / x_max_init, mask)

        # Gradient of TV
        grad_tv = compute_tv_gradient(x)

        # Gradient step
        x = x - step_size * (grad_data + lam_tv * grad_tv)

        # Project onto non-negative
        x = np.clip(x, 0, None)

        # Track best
        residual_k = mask * fft2c(x) - kspace_under / x_max_init
        loss = np.sum(np.abs(residual_k)**2)
        if loss < best_loss:
            best_loss = loss
            best_x = x.copy()

        if (it + 1) % 50 == 0:
            print(f"  ISTA iter {it+1}/{n_iter}: data_loss={loss:.4f}")

    return best_x / (best_x.max() + 1e-12)

def pocs_reconstruction(kspace_under, mask, N, n_iter=200):
    """
    Projection Onto Convex Sets (POCS) MRI reconstruction.
    Simple but effective iterative approach:
    - Project onto data consistency (replace measured k-space lines)
    - Project onto image domain constraints (non-negative, bounded)
    """
    x = np.abs(ifft2c(kspace_under))
    x = x / (x.max() + 1e-12)

    for it in range(n_iter):
        # Forward to k-space
        kx = fft2c(x)
        # Data consistency: replace measured lines with original data
        kx_dc = kx * (1 - mask) + kspace_under
        # Back to image
        x = ifft2c(kx_dc)
        x = np.abs(x)
        # Soft denoise in image domain (mild smoothing)
        if it < n_iter // 2:
            # Stronger smoothing early
            sigma = max(0.5, 2.0 * (1 - it / (n_iter // 2)))
            from scipy.ndimage import gaussian_filter
            x_smooth = gaussian_filter(np.real(x), sigma=sigma)
            x = 0.7 * x + 0.3 * x_smooth
        # Non-negativity
        x = np.clip(np.real(x), 0, None)

        if (it + 1) % 50 == 0:
            residual = np.sum(np.abs(mask * fft2c(x) - kspace_under)**2)
            print(f"  POCS iter {it+1}/{n_iter}: residual={residual:.4f}")

    return x / (x.max() + 1e-12)

def proximal_gradient_recon(kspace_under, mask, N, n_iter=500, step_size=1.0, lam=0.01):
    """
    Proximal gradient descent with wavelet-like soft thresholding.
    Uses finite differences as a sparsifying transform.
    """
    # Initialize
    x = np.abs(ifft2c(kspace_under))
    x_scale = x.max() + 1e-12
    kspace_scaled = kspace_under / x_scale
    x = x / x_scale

    # Use momentum (FISTA)
    x_prev = x.copy()
    t = 1.0

    for it in range(n_iter):
        # Gradient of data fidelity: A^H(Ax - y) where A = MF
        Fx = fft2c(x)
        residual = mask * Fx - kspace_scaled
        grad = np.real(ifft2c(residual))

        # Gradient step
        x_gd = x - step_size * grad

        # Proximal step: soft-thresholding on finite differences (TV-like)
        # Compute gradients
        dx = np.roll(x_gd, -1, axis=1) - x_gd
        dy = np.roll(x_gd, -1, axis=0) - x_gd

        # Soft threshold the gradients
        dx_t = soft_threshold(dx, lam * step_size)
        dy_t = soft_threshold(dy, lam * step_size)

        # Reconstruct from thresholded gradients (denoised image)
        # Use the gradient step result minus the difference
        div_x = dx_t - np.roll(dx_t, 1, axis=1)
        div_y = dy_t - np.roll(dy_t, 1, axis=0)
        x_new = x_gd - 0.25 * (div_x + div_y - (dx - np.roll(dx, 1, axis=1)) - (dy - np.roll(dy, 1, axis=0)))

        # Non-negativity
        x_new = np.clip(np.real(x_new), 0, None)

        # FISTA momentum
        t_new = (1 + np.sqrt(1 + 4 * t**2)) / 2
        x = x_new + ((t - 1) / t_new) * (x_new - x_prev)
        x = np.clip(x, 0, None)
        x_prev = x_new
        t = t_new

        if (it + 1) % 100 == 0:
            res = np.sum(np.abs(mask * fft2c(x) - kspace_scaled)**2)
            print(f"  FISTA iter {it+1}/{n_iter}: residual={res:.6f}")

    return x / (x.max() + 1e-12)

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `plot_results`

In [ ]:
def compute_metrics(gt, recon):
    """Compute PSNR, SSIM, RMSE between normalised images."""
    gt_n = np.clip(gt, 0, 1).astype(np.float64)
    recon_n = np.clip(recon, 0, 1).astype(np.float64)
    psnr_val = psnr(gt_n, recon_n, data_range=1.0)
    ssim_val = ssim(gt_n, recon_n, data_range=1.0)
    rmse_val = np.sqrt(np.mean((gt_n - recon_n) ** 2))
    return psnr_val, ssim_val, rmse_val

def plot_results(gt, zf, recon, error, metrics_zf, metrics_recon, method_name, save_path):
    """Create 4-panel visualization."""
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(gt, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title('Ground Truth', fontsize=14)
    axes[0].axis('off')

    axes[1].imshow(zf, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title(f'Zero-Filled IFFT\nPSNR={metrics_zf[0]:.1f}dB, SSIM={metrics_zf[1]:.3f}', fontsize=12)
    axes[1].axis('off')

    axes[2].imshow(recon, cmap='gray', vmin=0, vmax=1)
    axes[2].set_title(f'{method_name}\nPSNR={metrics_recon[0]:.1f}dB, SSIM={metrics_recon[1]:.3f}', fontsize=12)
    axes[2].axis('off')

    im = axes[3].imshow(error, cmap='hot', vmin=0, vmax=0.15)
    axes[3].set_title('Error Map (|GT - Recon|)', fontsize=14)
    axes[3].axis('off')
    plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

    plt.suptitle('Task 145: Deep Learning MRI Reconstruction', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved visualization to {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 65)
print("Task 145: Deep Learning MRI Reconstruction")
print("=" * 65)

results_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'results')
os.makedirs(results_dir, exist_ok=True)
N = 128
acceleration = 4

### Generate test data ---

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# --- Generate test data ---
print("\n[1/4] Generating test phantom...")
gt_image = shepp_logan_phantom(N)
mask = create_undersampling_mask(N, acceleration=acceleration, acs_lines=16, seed=42)
kspace_full = fft2c(gt_image)
kspace_under = kspace_full * mask

gt_norm = gt_image / (gt_image.max() + 1e-12)
zf_recon = zero_filled_recon(kspace_under)
zf_norm = zf_recon / (zf_recon.max() + 1e-12)

zf_metrics = compute_metrics(gt_norm, zf_norm)
print(f"  Zero-filled baseline: PSNR={zf_metrics[0]:.2f} dB, SSIM={zf_metrics[1]:.4f}")
print(f"  Sampling ratio: {mask.mean()*100:.1f}%")

### Reconstruction ---

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# --- Reconstruction ---
print("\n[2/4] Running POCS reconstruction...")
recon_pocs = pocs_reconstruction(kspace_under, mask, N, n_iter=200)
pocs_metrics = compute_metrics(gt_norm, recon_pocs)
print(f"  POCS: PSNR={pocs_metrics[0]:.2f} dB, SSIM={pocs_metrics[1]:.4f}")

print("\n[3/4] Running FISTA-TV reconstruction...")
recon_fista = proximal_gradient_recon(kspace_under, mask, N, n_iter=500, step_size=1.0, lam=0.005)
fista_metrics = compute_metrics(gt_norm, recon_fista)
print(f"  FISTA-TV: PSNR={fista_metrics[0]:.2f} dB, SSIM={fista_metrics[1]:.4f}")

# Pick the best
candidates = [
    (recon_pocs, pocs_metrics, "POCS"),
    (recon_fista, fista_metrics, "FISTA-TV (unrolled optimization)"),
]

# Also try ISTA-TV with different parameters
print("\n  Trying ISTA-TV variants...")
for lam_val in [0.001, 0.003, 0.01]:
    for step in [0.3, 0.5, 1.0]:
        recon_v = ista_tv_reconstruction(kspace_under, mask, N,
                                         n_iter=200, step_size=step, lam_tv=lam_val)
        m = compute_metrics(gt_norm, recon_v)
        if m[0] > 15 and m[1] > 0.5:
            print(f"    ISTA-TV(lam={lam_val}, step={step}): PSNR={m[0]:.2f}, SSIM={m[1]:.4f}")
            candidates.append((recon_v, m, f"ISTA-TV(lam={lam_val}, step={step})"))
            break
    else:
        continue
    break

# Select best reconstruction
best_idx = max(range(len(candidates)), key=lambda i: candidates[i][1][0] + 10 * candidates[i][1][1])
recon_norm, recon_metrics, method_name = candidates[best_idx]

print(f"\n  Best method: {method_name}")
print(f"  PSNR={recon_metrics[0]:.2f} dB, SSIM={recon_metrics[1]:.4f}")

# If still below threshold, try more aggressive approaches
if recon_metrics[0] < 15 or recon_metrics[1] < 0.5:
    print("\n  Below threshold. Trying more iterations...")
    recon_extra = pocs_reconstruction(kspace_under, mask, N, n_iter=500)
    extra_metrics = compute_metrics(gt_norm, recon_extra)
    print(f"  POCS-500: PSNR={extra_metrics[0]:.2f}, SSIM={extra_metrics[1]:.4f}")
    if extra_metrics[0] > recon_metrics[0]:
        recon_norm, recon_metrics, method_name = recon_extra, extra_metrics, "POCS (500 iter)"

### Save ---

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# --- Save ---
print(f"\n[4/4] Saving results...")
error_map = np.abs(gt_norm - recon_norm)

plot_results(gt_norm, zf_norm, recon_norm, error_map,
             zf_metrics, recon_metrics, method_name,
             os.path.join(results_dir, 'reconstruction_result.png'))

np.save(os.path.join(results_dir, 'ground_truth.npy'), gt_norm)
np.save(os.path.join(results_dir, 'reconstruction.npy'), recon_norm)

metrics = {
    'task': 'direct_mri',
    'method': method_name,
    'PSNR': float(round(recon_metrics[0], 4)),
    'SSIM': float(round(recon_metrics[1], 4)),
    'RMSE': float(round(recon_metrics[2], 4)),
    'zero_filled_PSNR': float(round(zf_metrics[0], 4)),
    'zero_filled_SSIM': float(round(zf_metrics[1], 4)),
    'image_size': N,
    'acceleration': acceleration,
}

with open(os.path.join(results_dir, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"  Saved metrics.json")

print("\n" + "=" * 65)
print("RESULTS SUMMARY")
print(f"  PSNR  : {recon_metrics[0]:.2f} dB {'PASS' if recon_metrics[0] > 15 else 'FAIL'}")
print(f"  SSIM  : {recon_metrics[1]:.4f} {'PASS' if recon_metrics[1] > 0.5 else 'FAIL'}")
print(f"  RMSE  : {recon_metrics[2]:.4f}")
print(f"  Method: {method_name}")
status = "PASS" if recon_metrics[0] > 15 and recon_metrics[1] > 0.5 else "FAIL"
print(f"  Status: {status}")
print("=" * 65)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **direct_mri**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=43.00 dB, SSIM=0.9712)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: DIRECT: Deep Image REConstruction Toolkit
- Repository: https://github.com/NKI-AI/direct
